In [5]:
import io
import sys
import pandas as pd
import numpy as np

from langchain_ollama import ChatOllama

from rich.console import Console
from rich.table import Table
from rich.panel import Panel
from rich.syntax import Syntax
from rich import box


console = Console()


# ============================================================
# Load Dataset
# ============================================================

df = pd.read_csv("dataset.csv")

console.print(
    "\n[bold green]Dataset loaded successfully[/bold green]\n"
)

table = Table(
    title="Dataset Preview",
    box=box.DOUBLE_EDGE
)

for col in df.columns:
    table.add_column(str(col))

for _, row in df.iterrows():
    table.add_row(*[str(i) for i in row])

console.print(table)


# ============================================================
# Ollama LLM
# ============================================================

# Make sure Ollama is running and llama3.1 is installed:
# ollama pull llama3.1

llm = ChatOllama(
    model="llama3.1",
    temperature=0
)


# ============================================================
# Agent 1 - Guiding Agent
# ============================================================

tasks = [
    "Display the entire dataset",

    "Print number of rows, number of columns and column names",

    "Identify and print quantitative and qualitative columns",

    "Detect missing values, print number of missing values and handle them",

    "Detect outliers, print high and low outliers and handle them using IQR",

    "Find duplicates and print number of duplicate rows",

    "Check the data types of all columns and convert columns to appropriate data types where necessary",

    "Detect highly skewed numerical columns and apply an appropriate transformation such as log1p where appropriate",

    "Detect columns with constant or near-zero variance and identify them as candidates for removal",
]


# ============================================================
# Agent 2 - Coding Agent
# ============================================================

def coder_agent(task):

    prompt = f"""
You are an expert Python data preprocessing engineer.

You are part of a multi-agent data preprocessing system.

The dataframe is already loaded and is named df.

Rules:

1. DO NOT reload the dataset.

2. Use pandas and numpy.

3. Return ONLY executable Python code.

4. DO NOT use markdown.

5. DO NOT use ```python.

6. DO NOT explain the code.

7. Any result that should be displayed must use print().

8. Modify df directly when preprocessing is required.

9. Numeric columns must be identified using:
   df.select_dtypes(include=np.number)

10. Categorical columns must be identified using:
    df.select_dtypes(include="object")

11. NEVER calculate df.mean() on the entire dataframe.

12. For missing numeric values:
    - Handle each numeric column separately.
    - Use the column mean only for numeric columns.

13. For missing categorical/object values:
    - Handle each categorical column separately.
    - Use "missing_value" for missing categorical values.

14. Do NOT use:
    df.fillna(df.mean(), inplace=True)

15. Do NOT use a fake column name such as:
    df['categorical_column']

16. For IQR outlier handling:
    - Identify numeric columns.
    - Calculate Q1 for each numeric column.
    - Calculate Q3 for each numeric column.
    - Calculate IQR = Q3 - Q1.
    - Lower bound = Q1 - 1.5 * IQR.
    - Upper bound = Q3 + 1.5 * IQR.
    - Print the number of low and high outliers.
    - Handle outliers using clipping rather than deleting rows.
    - Use:
      df[col] = df[col].clip(
          lower=lower_bound,
          upper=upper_bound
      )

17. For duplicates:
    - Print the number of duplicate rows.
    - Remove duplicates using:
      df.drop_duplicates(inplace=True)

18. For data type checking:
    - Print the current data type of every column.
    - Detect date-like columns and convert them to datetime when appropriate.
    - Detect numeric-looking columns and convert them to numeric when appropriate.
    - Do NOT convert SKU, ID, product codes, or identifiers into numeric columns merely because they contain numbers.
    - Preserve categorical and text columns.
    - Print the final data types after conversion.

19. For skewness:
    - Only analyze numerical columns.
    - Calculate skewness using:
      df[col].skew()
    - A column is considered highly skewed when:
      abs(skewness) > 1
    - Print the skewness of numerical columns.
    - For highly positively skewed numerical columns containing only non-negative values, apply:
      np.log1p(df[col])
    - Do NOT apply log1p to columns containing negative values.
    - Do NOT transform identifier columns.
    - Print which columns were transformed.
    - Do not transform a column more than once.

20. For constant and near-zero variance:
    - Only analyze numerical columns.
    - Identify constant columns where:
      df[col].nunique() <= 1
    - Identify near-zero variance columns using an appropriate threshold.
    - Print the constant columns.
    - Print the near-zero variance columns.
    - Do NOT automatically remove these columns.
    - Only identify them as candidates for removal.

21. Be careful with:
    - dates
    - SKU
    - ID
    - product codes
    - categorical columns
    - text columns

22. Do not perform mathematical operations on text, date, SKU, ID,
    product code, or categorical columns.

23. Before applying a transformation, verify that the column is numeric.

24. Do not create fake column names.

25. Clearly print what preprocessing was performed.\

26. For constant and near-zero variance detection:
    - Only analyze numerical columns.
    - Use:
      numeric_cols = df.select_dtypes(include=np.number).columns
    - Check each numerical column individually using a for loop.
    - A constant column has:
      df[col].nunique() <= 1
    - A near-zero variance column has:
      df[col].var() < 0.01
    - Print the column name and variance.
    - Print constant columns.
    - Print near-zero variance columns.
    - Identify them as candidates for removal.
    - Do NOT automatically remove them.
    - Do NOT use expressions such as:
      df.nunique(<= 1)
    - Do NOT use:
      numeric_cols[numeric_cols.nunique() < 0.01]

Task:
{task}
"""

    response = llm.invoke(prompt)

    code = response.content.strip()

    # Remove markdown fences if Ollama returns them
    code = code.replace("```python", "")
    code = code.replace("```", "")
    code = code.strip()

    console.print(
        "\n[bold yellow]Generated Code[/bold yellow]\n"
    )

    syntax = Syntax(
        code,
        "python",
        theme="monokai",
        line_numbers=False
    )

    console.print(syntax)

    return code


# ============================================================
# Agent 3 - Executor Agent
# ============================================================

def executor_agent(code):

    global df

    old_stdout = sys.stdout
    buffer = io.StringIO()

    try:

        local_vars = {
            "df": df,
            "np": np,
            "pd": pd
        }

        # Capture print output
        sys.stdout = buffer

        exec(
            code,
            {
                "pd": pd,
                "np": np
            },
            local_vars
        )

        # Restore stdout
        sys.stdout = old_stdout

        # Retrieve modified dataframe
        df = local_vars.get("df", df)

        output = buffer.getvalue()

        console.print(
            Panel(
                output if output else "Execution Completed",
                title="Execution Output",
                style="bold cyan on black"
            )
        )

        return True

    except Exception as e:

        sys.stdout = old_stdout

        console.print(
            Panel(
                f"{type(e).__name__}: {str(e)}",
                title="Execution Error",
                style="bold red"
            )
        )

        return False


# ============================================================
# Agent 4 - Checker Agent
# ============================================================

def checker_agent(iteration):

    if iteration < len(tasks):

        console.print(
            "\n[bold cyan]"
            "Checker Agent --> Pending Preprocessing"
            "[/bold cyan]"
        )

    else:

        console.print(
            "\n[bold green]"
            "Checker Agent --> Preprocessed Completely"
            "[/bold green]"
        )


# ============================================================
# Multi-Agent Execution
# ============================================================

for i, task in enumerate(tasks, start=1):

    console.print(
        f"\n[bold magenta]"
        f"====================================\n"
        f"Iteration {i}\n"
        f"===================================="
        f"[/bold magenta]"
    )

    console.print(
        f"[bold blue]"
        f"Guiding Agent --> {task}"
        f"[/bold blue]"
    )

    # --------------------------------------------------------
    # Coding Agent
    # --------------------------------------------------------

    code = coder_agent(task)

    # --------------------------------------------------------
    # Executor Agent
    # --------------------------------------------------------

    success = executor_agent(code)

    # --------------------------------------------------------
    # Checker Agent
    # --------------------------------------------------------

    checker_agent(i)

    if not success:

        console.print(
            "\n[bold red]"
            f"Iteration {i} failed. "
            "Continuing with the next iteration."
            "[/bold red]"
        )


# ============================================================
# Final Dataset Information
# ============================================================

console.print(
    "\n[bold green]Final Cleaned Dataset[/bold green]\n"
)

console.print(
    f"Rows: {len(df)}"
)

console.print(
    f"Columns: {len(df.columns)}"
)

console.print(
    f"Column names: {list(df.columns)}"
)


# ============================================================
# Final Cleaned Dataset Table
# ============================================================

final_table = Table(
    title="Processed Dataset",
    box=box.DOUBLE_EDGE
)

for col in df.columns:
    final_table.add_column(str(col))

for _, row in df.iterrows():
    final_table.add_row(
        *[str(i) for i in row]
    )

console.print(final_table)


# ============================================================
# Final Missing Value Check
# ============================================================

console.print(
    "\n[bold yellow]Final Missing Value Check[/bold yellow]"
)

missing_values = df.isnull().sum()

console.print(
    missing_values.to_string()
)


# ============================================================
# Final Duplicate Check
# ============================================================

console.print(
    "\n[bold yellow]Final Duplicate Check[/bold yellow]"
)

duplicate_count = df.duplicated().sum()

console.print(
    f"Duplicate rows remaining: {duplicate_count}"
)


# ============================================================
# Save Dataset
# ============================================================

output_file = "Assignment_Final_Dataset_1.csv"

df.to_csv(
    output_file,
    index=False
)

console.print(
    f"\n[bold green]"
    f"Processed dataset saved to {output_file}"
    f"[/bold green]"
)

Dataset loaded successfully

                        Dataset Preview                        
╔════════════╤═════════╤══════════╤═══════════════╤═══════════╗
║ date       │ sku     │ location │ quantity_sold │ inventory ║
╟────────────┼─────────┼──────────┼───────────────┼───────────╢
║ 2024-01-01 │ SKU_101 │ Chennai  │ 52.0          │ 120       ║
║ 2024-01-02 │ SKU_101 │ Chennai  │ 49.0          │ 115       ║
║ 2024-01-03 │ SKU_101 │ Chennai  │ nan           │ 110       ║
║ 2024-01-04 │ SKU_101 │ Chennai  │ 55.0          │ 105       ║
║ 2024-01-05 │ SKU_101 │ Chennai  │ 60.0          │ 100       ║
║ 2024-01-06 │ SKU_101 │ Chennai  │ -5.0          │ 95        ║
║ 2024-01-07 │ SKU_101 │ Chennai  │ 58.0          │ 90        ║
║ 2024-01-08 │ SKU_101 │ Chennai  │ 62.0          │ 85        ║
║ 2024-01-09 │ SKU_101 │ Chennai  │ 59.0          │ 80        ║
║ 2024-01-10 │ SKU_101 │ Chennai  │ 57.0          │ 75        ║
║ 2024-01-11 │ SKU_101 │ Chennai  │ 300.0         │ 70        ║
║ 2024-01-12 │ SKU_101 │ Chennai  │ 56.0          │ 65        ║
║ 2024-01-13 │ SKU_101 │ Chennai  │ 54.0          │ 60        ║
║ 2024-01-14 │ SKU_101 │ Chennai  │ 2.0           │ 55        ║
║ 2024-01-15 │ SKU_101 │ Chennai  │ 53.0          │ 50        ║
║ 2024-01-16 │ SKU_101 │ Chennai  │ 61.0          │ 45        ║
║ 2024-01-17 │ SKU_101 │ Chennai  │ 63.0          │ 40        ║
║ 2024-01-18 │ SKU_101 │ Chennai  │ 65.0          │ 35        ║
║ 2024-01-19 │ SKU_101 │ Chennai  │ nan           │ 30        ║
║ 2024-01-20 │ SKU_101 │ Chennai  │ 64.0          │ 25        ║
║ 2024-01-21 │ SKU_101 │ Chennai  │ 70.0          │ 20        ║
║ 2024-01-22 │ SKU_101 │ Chennai  │ 72.0          │ 15        ║
║ 2024-01-23 │ SKU_101 │ Chennai  │ 68.0          │ 10        ║
║ 2024-01-24 │ SKU_101 │ Chennai  │ 500.0         │ 5         ║
║ 2024-01-25 │ SKU_101 │ Chennai  │ 66.0          │ 0         ║
║ 2024-01-26 │ SKU_101 │ Chennai  │ 67.0          │ 0         ║
║ 2024-01-27 │ SKU_101 │ Chennai  │ 69.0          │ 0         ║
║ 2024-01-28 │ SKU_101 │ Chennai  │ 71.0          │ 10        ║
║ 2024-01-29 │ SKU_101 │ Chennai  │ 73.0          │ 15        ║
║ 2024-01-30 │ SKU_101 │ Chennai  │ 74.0          │ 20        ║
╚════════════╧═════════╧══════════╧═══════════════╧═══════════╝

====================================
Iteration 1
====================================

Guiding Agent --> Display the entire dataset

Generated Code

...

Final Missing Value Check

date             0
sku              0
location         0
quantity_sold    0
inventory        0

Final Duplicate Check

Duplicate rows remaining: 0

Processed dataset saved to Assignment_Final_Dataset_1.csv